<div style="background:linear-gradient(135deg,#050816,#101827 55%,#17324d);color:white;padding:28px;border-radius:18px;border:1px solid #28445f">
  <h1 style="margin:0;font-size:34px">Agente IA que simula 10.000 futuros</h1>
  <p style="font-size:18px;margin:8px 0 0 0;color:#b7d7ff">Caso practico de negocio: contrafactuales, Monte Carlo y recomendacion ejecutiva</p>
</div>

## Objetivo del caso

### Pregunta de negocio

Este proyecto compara cuatro decisiones de crecimiento para responder una pregunta muy concreta: que iniciativas deberia priorizar una empresa si quiere aumentar resultado economico sin asumir un nivel de riesgo excesivo.

Las cuatro opciones que se analizan son estas:

1. Mejorar landing + CTA + lead magnet.
2. Duplicar inversion en Ads.
3. Hacer un webinar de ventas.
4. Crear un nuevo producto.

### Como leer este notebook

El notebook esta pensado para seguir el analisis como una secuencia funcional sencilla:

1. Primero se carga la tabla historica del negocio.
2. Despues se resume la situacion actual por canal.
3. Luego se muestran los contrafactuales, es decir, el efecto esperado medio de cada decision si se aplicara sobre el historico.
4. A continuacion se ejecuta una simulacion Monte Carlo en tiempo real para introducir incertidumbre y medir riesgo.
5. Por ultimo, un agente resume todo y emite una recomendacion ejecutiva.

### Archivos del proyecto

- `scripts/simulacion_montecarlo.py`: genera los datos, estima los efectos de cada decision y ejecuta la simulacion.
- `scripts/agente_montecarlo.py`: concentra las funciones auxiliares y la capa de agente que interpreta los resultados.

In [ ]:
import pandas as pd
import plotly.express as px

from scripts.agente_montecarlo import (
    LIVE_DASHBOARD_PATH,
    cargar_artifacts,
    ejecutar_simulacion_montecarlo,
    estado_simulacion_montecarlo,
    recargar_resultados_montecarlo,
    resumen_contrafactuales,
    resumen_ejecutivo,
    resumen_por_canal,
    run_agent,
)

COLOR_FONDO = '#07111f'
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

## 1. Cargar el dataset

El analisis parte de una tabla historica donde cada fila representa una oportunidad comercial con su contexto, sus senales de marketing y su resultado economico final.

Piensa en este dataset como una vista analitica ya preparada para estudiar decisiones de negocio.

In [ ]:
artifacts = cargar_artifacts()
df = artifacts['dataset']
parametros_df = artifacts['parametros']
print(df.shape)
df.head(3)

## 2. Lectura rapida del negocio actual

Este paso resume el punto de partida por canal: volumen, conversion, ingresos, beneficio y calidad media del lead.

La idea es entender como rinde hoy el negocio antes de comparar decisiones futuras.

In [ ]:
resumen_canal = resumen_por_canal(df).reset_index()
resumen_canal

In [ ]:
fig = px.bar(
    resumen_canal,
    x='canal',
    y='beneficio_contribucion_eur',
    color='conversion',
    title='Beneficio de contribucion por canal y tasa de conversion',
    template='plotly_dark',
    color_continuous_scale='Turbo',
)
fig.update_layout(height=460, plot_bgcolor=COLOR_FONDO, paper_bgcolor=COLOR_FONDO)
fig.show()

## 3. Contrafactuales estimados desde el historico

Antes de introducir incertidumbre, el proyecto calcula el efecto esperado medio de cada palanca sobre el historico.

Un contrafactual responde a esta pregunta: que habria pasado con estas mismas oportunidades si la empresa hubiera aplicado una decision concreta, como mejorar el funnel o activar un webinar.

Este bloque muestra esa primera capa del analisis: el uplift esperado en conversion y en beneficio por oportunidad.

In [ ]:
contrafactuales = resumen_contrafactuales(parametros_df)
contrafactuales

## 4. Lanzar la simulacion Monte Carlo en tiempo real

Una vez estimado el efecto medio de cada decision, el siguiente paso es medir como cambia ese resultado cuando aparecen incertidumbre, retrasos y variabilidad de ejecucion.

Esta celda lanza la simulacion Monte Carlo en tiempo real y actualiza el dashboard live a medida que avanzan las iteraciones.

In [ ]:
lanzamiento = ejecutar_simulacion_montecarlo(
    simulaciones_nuevas=10000,
    modo='background',
    progress_every=100,
)
lanzamiento

In [ ]:
estado = estado_simulacion_montecarlo()
estado

## 5. Recargar resultados y leer el ranking

Cuando la simulacion termina, este paso vuelve a cargar los resultados generados y construye el ranking ejecutivo final.

Aqui ya no se mira solo el beneficio esperado, sino tambien la probabilidad de perdida y el perfil de riesgo de cada opcion.

In [ ]:
artifacts = recargar_resultados_montecarlo()
df = artifacts['dataset']
parametros_df = artifacts['parametros']
resumen_live = resumen_ejecutivo(artifacts['resumen'])
print(LIVE_DASHBOARD_PATH)
resumen_live

In [ ]:
fig = px.bar(
    resumen_live,
    x='decision',
    y='beneficio_esperado_eur',
    color='probabilidad_perdida',
    title='Ranking ejecutivo tras la Monte Carlo',
    template='plotly_dark',
    color_continuous_scale='RdYlGn_r',
)
fig.update_layout(height=480, plot_bgcolor=COLOR_FONDO, paper_bgcolor=COLOR_FONDO)
fig.show()

## 6. Pedir la recomendacion al agente

En este ultimo paso, el agente consulta los resultados del analisis y redacta una recomendacion ejecutiva.

Su papel no es recalcular el modelo, sino interpretar el ranking final, comparar retorno y riesgo y traducirlo a una decision comprensible para negocio.

La API key de OpenAI se toma del archivo `.env` incluido en la raiz del proyecto.

In [ ]:
PREGUNTA = (
    'Tenemos presupuesto para ejecutar solo dos de estas cuatro iniciativas en los proximos 6 meses: '
    '(1) mejorar el funnel completo -landing, CTA y lead magnet-, '
    '(2) hacer un webinar de ventas, '
    '(3) duplicar la inversion en Facebook Ads y Google Ads, '
    '(4) lanzar un nuevo producto. '
    'Analiza cada opcion en profundidad considerando tanto el retorno esperado como el riesgo, '
    'e indica cuales dos elegiras y por que descartarias las otras dos.'
)

respuesta = run_agent(PREGUNTA, model='gpt-4o-mini', verbose=True)
print(respuesta)

## Estructura del proyecto

- `caso_agente_ia_montecarlo.ipynb`: notebook principal para seguir la demo paso a paso.
- `scripts/`: codigo Python del pipeline y del agente.
- `datos/`: dataset sintetico y resultados persistidos de la simulacion.
- `dashboards/`: dashboard live y dashboard final.
- `.env`: archivo donde se define `OPENAI_API_KEY`.
- `README.md`: guia de instalacion y uso.

## Dashboards

- `dashboards/dashboard_live_montecarlo.html`: muestra la simulacion mientras se esta ejecutando.
- `dashboards/dashboard_nasa_montecarlo.html`: presenta una version final, pensada para visualizar el ranking de decisiones de forma mas impactante.